# 실습 2 — Text 분류 (14분)

교재: [docs/02-text.md](../docs/02-text.md)

네이버 영화 리뷰 감성 분석(NSMC)으로 **방법 A: TF-IDF + ML**을 끝까지 돌립니다.
2~3분이면 끝납니다.

> **실행 환경: Colab T4.** `런타임 → 런타임 유형 변경 → T4 GPU`로 먼저 바꾸고 시작하세요.

방법 B(BERT)는 T4에서도 20~40분이 걸려 강의 시간에 맞지 않습니다.
이 노트북에서는 **코드만 보고 넘어가고**, 실제로 돌려 볼 때는
[templates/text_classification.py](../templates/text_classification.py)의 방법 B 구간을 쓰세요.

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 0. 시험 환경 세팅

시험 시간은 **총 180분**입니다. 시작하면 세 가지를 먼저 합니다.
연습할 때부터 같은 순서로 하세요.

| | 언제 | 무엇을 |
|---|---|---|
| **세션당 한 번** | 시작 직후 | GPU 연결 확인, `PHONE`, 시드 |
| **문제마다** | 문제를 열 때 | `PROB_NO` 한 줄 |
| **계속** | 셀을 완성할 때마다 | `Ctrl+S` — 자동 저장이 안 됩니다 |

GPU 확인은 **세션 시작 때 한 번이면 끝**입니다. 문제마다 다시 하지 않습니다.
실전 템플릿으로는 [templates/00_session_start.py](../templates/00_session_start.py)에 모여 있습니다.

In [ ]:
# --- 세션당 한 번 ---
PHONE = '01012345678'      # 본인 휴대폰 번호 (하이픈 없이)

# --- 문제마다 ---
PROB_NO = 1

SUBMIT_CSV = f'{PHONE}_{PROB_NO}.csv'
SUBMIT_MODEL = f'{PHONE}_{PROB_NO}.pkl'
print(f"제출 파일: {SUBMIT_CSV} / {SUBMIT_MODEL}")

## 1. 로드

NSMC는 네이버 영화 리뷰 20만 건에 긍정/부정 라벨이 붙은 데이터입니다.
강의 시간에 맞추려고 3만 건만 샘플링합니다. (전체 15만 건으로 돌려도 몇 분이면 끝납니다.)

In [ ]:
NSMC_TRAIN = 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt'
NSMC_TEST = 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt'

df_train = pd.read_csv(NSMC_TRAIN, sep='\t').dropna()
df_test = pd.read_csv(NSMC_TEST, sep='\t').dropna()

# 강의용 샘플링 - 실제 시험에서는 이 두 줄을 지우세요
df_train = df_train.sample(30000, random_state=RANDOM_SEED).reset_index(drop=True)
df_test = df_test.sample(6000, random_state=RANDOM_SEED).reset_index(drop=True)

TEXT_COL, TARGET = 'document', 'label'
target_names = ['부정', '긍정']
NUM_CLASSES = df_train[TARGET].nunique()
label_map = {i: n for i, n in enumerate(target_names)}

print(f"Train {df_train.shape} / Test {df_test.shape}")
print(df_train[TARGET].value_counts().sort_index())
df_train.head()

## 2. EDA — 길이가 `MAX_LENGTH`를 정한다

BERT를 쓸 경우 `MAX_LENGTH`를 감으로 정하지 말고 이 분포를 보고 정합니다.

In [ ]:
lengths = df_train[TEXT_COL].astype(str).apply(len)
print(lengths.describe().round(1))
print(f"95 퍼센타일: {lengths.quantile(0.95):.0f}자")

lengths.hist(bins=50, figsize=(8, 3))
plt.xlabel('text length'); plt.title('Text length distribution')
plt.tight_layout(); plt.show()

95%가 110자 안쪽입니다. 한국어는 한 글자가 보통 1~2토큰이 되므로 BERT `MAX_LENGTH=128`이면 충분합니다.
여기에 256이나 512를 넣으면 **성능은 그대로인데 학습 시간만 2~4배**로 늘어납니다.

## 3. 정제

TF-IDF는 단어 등장 횟수만 보므로 문장부호가 사라져도 손해가 적습니다. 세게 정제해도 됩니다.
(BERT라면 반대로 약하게 하세요. `?`, `!`, 어미가 문맥 정보입니다.)

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)                  # HTML 태그
    text = re.sub(r'http\S+|www\.\S+', ' ', text)         # URL
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)      # 특수문자
    text = re.sub(r'\s+', ' ', text).strip()              # 다중 공백
    return text


print('정제 전:', df_train[TEXT_COL].iloc[3])
df_train[TEXT_COL] = df_train[TEXT_COL].apply(clean_text)
df_test[TEXT_COL] = df_test[TEXT_COL].apply(clean_text)
print('정제 후:', df_train[TEXT_COL].iloc[3])

In [ ]:
# 특수문자만으로 이루어진 리뷰(예: "ㅋㅋㅋ", "...")는 정제 후 빈 문자열이 된다.
# 그냥 두면 벡터화 단계에서 조용히 문제를 일으키므로 여기서 걸러낸다.
empty = (df_train[TEXT_COL].str.strip() == '').sum()
print(f"정제 후 빈 텍스트: {empty}건")

df_train = df_train[df_train[TEXT_COL].str.strip() != ''].reset_index(drop=True)
df_test = df_test[df_test[TEXT_COL].str.strip() != ''].reset_index(drop=True)
print(f"남은 데이터: train {len(df_train)} / test {len(df_test)}")

## 4. 분할

In [ ]:
df_tr, df_val = train_test_split(
    df_train, test_size=0.2, random_state=RANDOM_SEED, stratify=df_train[TARGET]
)
df_tr = df_tr.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
print(f"Train {len(df_tr)} / Val {len(df_val)} / Test {len(df_test)}")

## 5. TF-IDF 벡터화 (단어 단위)

파라미터 하나하나가 의미를 갖습니다.

In [ ]:
tfidf_word = TfidfVectorizer(
    max_features=10000,   # 상위 1만 피처만 (메모리/속도)
    ngram_range=(1, 2),   # 단어 + 2단어 조합
    min_df=2,             # 문서 1개에만 나오는 단어 제거 (오타 노이즈)
    max_df=0.95,          # 95% 이상 문서에 나오는 단어 제거 (변별력 없음)
    sublinear_tf=True,    # TF 에 log 스케일
)

Xw_train = tfidf_word.fit_transform(df_tr[TEXT_COL])   # fit 은 train 에만!
Xw_val = tfidf_word.transform(df_val[TEXT_COL])
y_train, y_val = df_tr[TARGET], df_val[TARGET]

print(f"TF-IDF 행렬: Train {Xw_train.shape} / Val {Xw_val.shape}")
print(f"희소도: {Xw_train.nnz / (Xw_train.shape[0] * Xw_train.shape[1]) * 100:.3f}% 만 0이 아님")

### `ngram_range=(1,2)`가 왜 필요한가

"재미없다"의 부정을 단어 하나로는 못 잡는 경우가 있습니다. 실제로 확인해 봅시다.

In [ ]:
sample = ['별로 재미 없다', '정말 재미 있다']
uni = TfidfVectorizer(ngram_range=(1, 1)).fit(sample)
bi = TfidfVectorizer(ngram_range=(1, 2)).fit(sample)
print('(1,1) 피처:', uni.get_feature_names_out().tolist())
print('(1,2) 피처:', bi.get_feature_names_out().tolist())

바이그램을 넣으면 `별로 재미`, `재미 없다` 같은 조합이 피처가 됩니다.
대신 피처 수가 몇 배로 늘어나므로 `max_features`로 상한을 겁니다.

## 6. 모델 비교

TF-IDF의 출력은 1만 차원 희소 행렬입니다. 이런 데이터에서는 **선형 모델이 강합니다.**
Tabular에서 주력이던 RandomForest는 여기서 느리고 성능도 밀려서 후보에도 넣지 않습니다.

In [ ]:
def compare_models(X_tr, X_va, y_tr, y_va, tag=''):
    """세 모델을 같은 조건으로 돌려 결과표와 1위 모델을 반환."""
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
        'LinearSVC': LinearSVC(max_iter=2000, random_state=RANDOM_SEED),
        'MultinomialNB': MultinomialNB(alpha=0.1),
    }
    rows, best = {}, (0, '', None)
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)
        acc = accuracy_score(y_va, pred)
        rows[name] = {
            'Accuracy': acc,
            'F1_macro': f1_score(y_va, pred, average='macro'),
            'CV_Acc': cross_val_score(model, X_tr, y_tr, cv=5).mean(),
        }
        if acc > best[0]:
            best = (acc, name, model)
    table = pd.DataFrame(rows).T.sort_values('Accuracy', ascending=False)
    print(f"[{tag}]")
    print(table.round(4))
    print(f"1위: {best[1]} ({best[0]:.4f})\n")
    return table, best


table_word, (acc_word, name_word, model_word) = compare_models(
    Xw_train, Xw_val, y_train, y_val, tag='단어 n-gram (1,2)'
)

## 7. 한국어의 함정 — 공백 토큰화는 약하다

0.78 근처가 나왔을 겁니다. 영어였다면 이 정도 데이터에서 0.88은 나옵니다. 왜 낮을까요?

`TfidfVectorizer`의 기본 토큰화는 **공백 기준**입니다. 그런데 한국어는 조사와 어미가 붙습니다.

```
"영화가"  "영화는"  "영화를"  "영화도"   →  전부 다른 단어로 취급
```

같은 뜻인데 피처가 4개로 흩어지니 학습이 어려워집니다.
제대로 하려면 형태소 분석기(KoNLPy, Mecab)를 쓰지만, **시험 환경에 설치돼 있으리라는 보장이 없습니다.**

설치 없이 쓸 수 있는 우회로가 있습니다. **문자 단위 n-gram**입니다.

In [ ]:
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',    # 단어 경계를 존중하는 문자 n-gram
    ngram_range=(2, 4),    # 2~4글자 조각
    max_features=50000,    # 문자 조합은 종류가 많으므로 상한을 올린다
    min_df=2,
    sublinear_tf=True,
)

Xc_train = tfidf_char.fit_transform(df_tr[TEXT_COL])
Xc_val = tfidf_char.transform(df_val[TEXT_COL])
print(f"문자 n-gram 행렬: Train {Xc_train.shape}")
print('피처 예시:', tfidf_char.get_feature_names_out()[5000:5008].tolist())

table_char, (acc_char, name_char, model_char) = compare_models(
    Xc_train, Xc_val, y_train, y_val, tag='문자 n-gram (2,4)'
)

In [ ]:
print(f"단어 n-gram 최고: {acc_word:.4f}  ({name_word})")
print(f"문자 n-gram 최고: {acc_char:.4f}  ({name_char})")
print(f"차이: {acc_char - acc_word:+.4f}")

**7%p 가까이 오릅니다.** 조사·어미가 붙어도 `재미없`, `최악이` 같은 조각은 그대로 잡히기 때문입니다.

대가는 계산량입니다. 피처가 5배로 늘어 벡터화가 3배쯤 느려집니다.
그래도 몇 초 수준이라 **시험에서는 거의 항상 이득**입니다.

> **기억할 것** — 한국어 텍스트에서 TF-IDF 점수가 낮게 나오면
> 모델을 바꾸기 전에 `analyzer='char_wb', ngram_range=(2,4)`를 먼저 시도하세요.

In [ ]:
# 더 좋은 쪽을 최종 채택
if acc_char >= acc_word:
    tfidf, best_model, best_name = tfidf_char, model_char, name_char
    X_train, X_val = Xc_train, Xc_val
else:
    tfidf, best_model, best_name = tfidf_word, model_word, name_word
    X_train, X_val = Xw_train, Xw_val

X_test = tfidf.transform(df_test[TEXT_COL])
print(f"채택: {best_name} on {'char' if tfidf is tfidf_char else 'word'} n-gram")

## 8. 평가

In [ ]:
y_pred_val = best_model.predict(X_val)
acc = accuracy_score(y_val, y_pred_val)
f1 = f1_score(y_val, y_pred_val, average='macro')

print(f"Accuracy {acc:.4f} / F1(macro) {f1:.4f}")
grade = next((g for t, g in [(0.95, '매우 우수'), (0.90, '우수'),
                             (0.85, '양호'), (0.80, '보통')] if acc >= t), '미흡')
print(f"-> {grade}\n")
print(classification_report(y_val, y_pred_val, target_names=target_names))

# Acc 는 높은데 F1 이 낮으면 다수 클래스만 맞히고 있다는 뜻
if acc - f1 > 0.1:
    print("Accuracy >> F1 : 클래스 불균형 -> class_weight='balanced' 검토")

### 모델이 무엇을 배웠는지 열어 보기

선형 모델의 장점은 계수를 그대로 읽을 수 있다는 것입니다.
여기서는 읽기 쉽도록 **단어 단위** 로지스틱 회귀의 계수를 봅니다.

In [ ]:
interp = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
interp.fit(Xw_train, y_train)

feature_names = np.array(tfidf_word.get_feature_names_out())
coefs = interp.coef_[0]

print('[긍정 신호 단어]')
print(', '.join(feature_names[np.argsort(coefs)[-15:][::-1]]))
print('\n[부정 신호 단어]')
print(', '.join(feature_names[np.argsort(coefs)[:15]]))

사람이 봐도 납득이 가는 단어들이 나옵니다.
여기에 엉뚱한 단어(특정 ID, 날짜 등)가 상위에 있으면 데이터에 누수나 편향이 있다는 신호입니다.

## 9. 제출

In [ ]:
y_pred_test = best_model.predict(X_test)

submit_df = pd.DataFrame({
    'id': df_test['id'],
    'predicted': y_pred_test,
    'predicted_name': pd.Series(y_pred_test).map(label_map),
})
submit_df.to_csv(SUBMIT_CSV, index=False)

# 텍스트는 벡터라이저가 없으면 모델만으로 예측할 수 없다. 함께 담아 저장한다.
import joblib, os
joblib.dump({'model': best_model, 'vectorizer': tfidf}, SUBMIT_MODEL)

print(f"[Test 성능] Accuracy {accuracy_score(df_test[TARGET], y_pred_test):.4f}")
print(f"[저장] {SUBMIT_CSV} / {SUBMIT_MODEL}")

assert len(submit_df) == len(df_test), '행 수 불일치!'
print('생성된 제출 파일:', [f for f in os.listdir('.') if f.startswith(PHONE)])
submit_df.head()

---
## 방법 B (BERT)는 여기서 코드만 확인

이 셀은 **실행하지 마세요.** GPU가 없으면 매우 오래 걸립니다.
핵심은 세 줄뿐이고, 나머지는 템플릿이 처리합니다.

```python
BERT_MODEL_NAME = 'klue/bert-base'   # 한국어 데이터에 영어 모델을 쓰면 성능이 무너진다
LEARNING_RATE   = 2e-5               # 협상 대상 아님. 1e-3 을 넣으면 사전학습 지식이 파괴된다

model = TFBertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=NUM_CLASSES,
    from_pt=True,                    # HF 한국어 모델은 PyTorch 가중치만 있는 경우가 많다
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  # softmax 전 출력
    metrics=['accuracy'],
)

pred_labels = np.argmax(model.predict(val_ds).logits, axis=1)   # .logits 에서 꺼낸다
```

BERT는 형태소 문제를 서브워드 토크나이저로 이미 해결해 두었기 때문에
NSMC에서 0.90 안팎까지 올라갑니다. 다만 20~40분이 듭니다.

**전략을 다시 확인하세요.** A를 먼저 돌려 `submission.csv`를 확보한 뒤,
점수가 부족하고 30분 이상 남았을 때만 B를 시도합니다.
B부터 시작했다가 OOM으로 죽으면 제출할 것이 하나도 없습니다.

---
## 혼자 해 볼 것

1. 문자 n-gram의 `ngram_range`를 `(2,5)`로 넓히면 점수가 더 오르나요? 시간은 얼마나 늘어나나요?
2. `clean_text`에서 특수문자 제거 줄을 빼면 어떻게 되나요? (한국어 리뷰의 `ㅋㅋ`, `ㅠㅠ`는 감성 신호입니다)
3. 샘플링 두 줄을 지우고 전체 15만 건으로 돌리면 점수가 오르나요?

다음: [실습 3 — Image](03_image_quickstart.ipynb) · 교재 [docs/03-image.md](../docs/03-image.md)